# Evaluate: plain Gemma 4 31B vs Gemma + VishwamAI LoRA adapter
Both models answer the **same** randomly sampled questions from the MMLU and MMLU-Pro **test** sets (used for evaluation only, never for training). Scoring: the model's probability for each answer letter right after the prompt; the highest wins.
Reports accuracy with a 95% Wilson interval and a paired comparison (how many questions only the tuned model got right vs only the base model). With 300 questions the interval is about +-5 points, so a small difference is not evidence of a gain.

**Setup on Kaggle:** Accelerator GPU T4 x2, Internet on, secret `HF_TOKEN`. Add your training notebook's output as an input (Add Input -> Notebook Output Files -> *Vishwamai QLORA*) so the adapter is found at `/kaggle/input/...`; or leave it if the adapter is in `/kaggle/working`.

In [ ]:
%pip -q install -U transformers peft bitsandbytes accelerate datasets huggingface_hub safetensors

In [ ]:
import os, sys, json, time, glob, shutil, math, random, subprocess
try:
    from google.colab import drive; drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/vishwamai'
except Exception:
    WORK = '/kaggle/working/vishwamai' if os.path.exists('/kaggle/working') else './vishwamai'
GDIR = f'{WORK}/gemma'; os.makedirs(GDIR, exist_ok=True)

MODEL_ID = 'google/gemma-4-31B'            # base weights. Try 'google/gemma-4-E4B' first if you only have a small GPU.
ADAPTER_DIR = f'{GDIR}/adapter'; SFT_FILE = f'{GDIR}/sft.jsonl'
USE_SWE_SMITH_TRAJECTORIES = False          # see the warning above before turning this on
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'    # must be set before torch touches the GPU (restart the session after changing)
MAX_LEN = 512;      # T4s have 15 GB each: keep sequences short (raise on bigger GPUs)
LORA_R = 16; LORA_ALPHA = 32; LR = 1e-4
TOPIC = 'eval'                              # today's topic, see TOPICS in the data cell
MAX_EXAMPLES_PER_SOURCE = 20000
CACHE_DIR = '/tmp/hf_cache'            # model download goes here and is deleted after loading (needs ~65 GB free)
TIME_BUDGET_MIN = 180; BATCH_TOKENS_ACCUM = 16   # sequences per optimiser step (micro-batch is 1 sequence)
N_MMLU = 300; N_PRO = 300           # questions sampled from each test set (fixed seed, same for base and tuned)
MAX_EVAL_TOKENS = 1024              # longer questions are skipped (for both models)

In [ ]:
def get_hf_token(required=False):
    t = os.environ.get('HF_TOKEN')
    if not t:
        try:
            from google.colab import userdata; t = userdata.get('HF_TOKEN')
        except Exception: pass
    if not t:
        try:
            from kaggle_secrets import UserSecretsClient; t = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception: pass
    if not t:
        try:
            from huggingface_hub import get_token; t = get_token()
        except Exception: pass
    if not t and required:
        from getpass import getpass; t = getpass('Hugging Face token (hidden): ')
    return t or None
HF_TOKEN = get_hf_token()

In [ ]:
# Guard: refuse to start on GPUs / disk that cannot work. Works for 1 big GPU or 2 x T4 (Kaggle: Accelerator -> GPU T4 x2).
import torch
assert torch.cuda.is_available(), 'No GPU: enable one in the notebook settings.'
n_gpu = torch.cuda.device_count(); gpus = [torch.cuda.get_device_properties(i) for i in range(n_gpu)]
total_gb = sum(g.total_memory for g in gpus) / 1e9
os.makedirs(CACHE_DIR, exist_ok=True); free_disk = shutil.disk_usage(CACHE_DIR).free / 1e9
BF16 = torch.cuda.is_bf16_supported() and all(g.major >= 8 for g in gpus)
print(f'{n_gpu} GPU(s): ' + ', '.join(f'{g.name} {g.total_memory/1e9:.0f} GB' for g in gpus) + f' | total {total_gb:.0f} GB | free disk at {CACHE_DIR}: {free_disk:.0f} GB | bf16 {BF16}')
big = '31B' in MODEL_ID or '26B' in MODEL_ID
if big:
    assert total_gb >= 28, f'{MODEL_ID} in 4-bit needs about 20 GB for weights plus activations: use 2 x T4 or a 40 GB GPU (found {total_gb:.0f} GB total).'
    assert free_disk >= 66, f'the download is ~63 GB and {CACHE_DIR} has {free_disk:.0f} GB free. Kaggle /kaggle/working is too small; try a different CACHE_DIR or a Colab A100 session.'

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, cache_dir=CACHE_DIR)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16)
DT = torch.bfloat16 if BF16 else torch.float16
# GPU 0 also holds the embedding / output layer, whose logits are huge (262k vocabulary): give it fewer layers than the others
max_mem = {i: f'{int(g.total_memory/2**30) - (7 if (i == 0 and n_gpu > 1) else 2)}GiB' for i, g in enumerate(gpus)}
max_mem['cpu'] = '24GiB'
kw = dict(quantization_config=bnb, device_map='auto', max_memory=max_mem, token=HF_TOKEN, torch_dtype=DT, cache_dir=CACHE_DIR)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
except Exception as e:
    print('AutoModelForCausalLM failed:', type(e).__name__, str(e)[:200], '\n-> trying AutoModelForImageTextToText')
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kw)
print(type(model).__name__, '| GPU memory used (GB):', [round(torch.cuda.memory_allocated(i)/1e9, 1) for i in range(n_gpu)])
print('devices:', sorted({str(v) for v in getattr(model, 'hf_device_map', {}).values()}))
shutil.rmtree(CACHE_DIR, ignore_errors=True)     # frees ~63 GB; the weights now live in GPU memory
print('download cache removed; free disk now', round(shutil.disk_usage('/').free/1e9), 'GB')

In [ ]:
# Find the adapter and attach it. Order: ADAPTER_PATH (a folder you set) -> ADAPTER_REPO (Hugging Face) -> files already here
# (this session / attached inputs / WORK) -> download the public Kaggle notebook output (kagglehub, then the kaggle CLI).
ADAPTER_PATH = ''                          # e.g. '/kaggle/input/vishwamai-qlora/vishwamai/gemma/adapter'
ADAPTER_REPO = ''                          # e.g. 'VishwamAI/vishwamai-gemma4-31b-lora-v0' (private is fine; needs HF_TOKEN)
KAGGLE_NOTEBOOK = 'bmentech/vishwamai-qlora'
from peft import PeftModel
def find_adapter(root):
    hits = glob.glob(f'{root}/**/adapter_config.json', recursive=True)
    return os.path.dirname(hits[0]) if hits else None
ADAPTER = None
if ADAPTER_PATH:
    if os.path.exists(f'{ADAPTER_PATH}/adapter_config.json'): ADAPTER = ADAPTER_PATH
    else: print(f'ADAPTER_PATH has no adapter_config.json ({ADAPTER_PATH}) - ignoring it and searching instead')
if not ADAPTER and ADAPTER_REPO:
    from huggingface_hub import snapshot_download
    ADAPTER = snapshot_download(ADAPTER_REPO, token=HF_TOKEN, allow_patterns=['adapter_*'])
for root in (ADAPTER_DIR, '/kaggle/input', WORK):
    if not ADAPTER and os.path.exists(root): ADAPTER = find_adapter(root)
if not ADAPTER and KAGGLE_NOTEBOOK:
    print('adapter not found locally; downloading the output of', KAGGLE_NOTEBOOK)
    dest = '/kaggle/working/adapter_dl'; os.makedirs(dest, exist_ok=True)
    try:
        import kagglehub
        ADAPTER = find_adapter(kagglehub.notebook_output_download(KAGGLE_NOTEBOOK))
    except Exception as e:
        print('  kagglehub failed:', type(e).__name__, str(e)[:200])
    if not ADAPTER:
        r = subprocess.run(['kaggle', 'kernels', 'output', KAGGLE_NOTEBOOK, '-p', dest], capture_output=True, text=True)
        print('  kaggle CLI:', (r.stdout + r.stderr)[-400:]); ADAPTER = find_adapter(dest)
if not ADAPTER:
    print('/kaggle/input contains:', [os.path.join(d, x) for d, _, fs in os.walk('/kaggle/input') for x in fs][:30] if os.path.exists('/kaggle/input') else 'nothing')
    raise RuntimeError('adapter not found. Use Add Input -> your vishwamai-qlora notebook output, or set ADAPTER_PATH / ADAPTER_REPO, then re-run this cell.')
print('adapter:', ADAPTER)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval(); model.config.use_cache = True

In [ ]:
import random, math
from datasets import load_dataset
tok.truncation_side = 'left'
if not getattr(tok, 'chat_template', None):          # same template logic as in training
    try: tok.chat_template = AutoTokenizer.from_pretrained(MODEL_ID + '-it', token=HF_TOKEN, cache_dir=CACHE_DIR).chat_template; print('chat template from -it')
    except Exception as e:
        print('using simple template:', type(e).__name__)
        tok.chat_template = ("{{ bos_token }}{% for m in messages %}<start_of_turn>{{ 'model' if m['role'] == 'assistant' else m['role'] }}\n"
                             "{{ m['content'] }}<end_of_turn>\n{% endfor %}{% if add_generation_prompt %}<start_of_turn>model\n{% endif %}")
LET = 'ABCDEFGHIJ'
# Every spelling of a letter the model might emit first: 'A', ' A', '▁A' (SentencePiece space marker). The score for a letter is the
# log-sum-exp of its spellings, so a model that prefers ' A' over 'A' is not penalised for a formatting habit (this made the first run unfair to the base model).
unk = getattr(tok, 'unk_token_id', None); let_var = []
for c in LET:
    ids = set()
    for form in (c, ' ' + c):
        e = tok.encode(form, add_special_tokens=False)
        if e: ids.add(e[-1])
    t = tok.convert_tokens_to_ids('▁' + c)
    if t is not None and t != unk: ids.add(t)
    let_var.append(sorted(ids))
print('letter token ids:', {LET[i]: let_var[i] for i in range(4)})

def build(q, opts):
    body = q + '\n' + '\n'.join(f'{LET[i]}. {o}' for i, o in enumerate(opts)) + '\nAnswer with the letter.'
    return tok.apply_chat_template([{'role': 'user', 'content': body}], tokenize=False, add_generation_prompt=True)

def sample(ds, n, to_item, seed=0):
    idx = list(range(len(ds))); random.Random(seed).shuffle(idx); out = []
    for i in idx:
        q, opts, gold = to_item(ds[i]); text = build(q, opts)
        ids = tok(text, add_special_tokens=False, return_tensors='pt').input_ids
        if ids.shape[1] <= MAX_EVAL_TOKENS: out.append((ids, len(opts), gold))
        if len(out) >= n: break
    return out

mmlu = load_dataset('cais/mmlu', 'all', split='test', token=HF_TOKEN)
items_mmlu = sample(mmlu, N_MMLU, lambda ex: (ex['question'], ex['choices'], ex['answer']))
try:
    pro = load_dataset('TIGER-Lab/MMLU-Pro', split='test', token=HF_TOKEN)
    items_pro = sample(pro, N_PRO, lambda ex: (ex['question'], ex['options'], ex['answer_index']), seed=1)
except Exception as e:
    print('MMLU-Pro not loaded:', type(e).__name__, str(e)[:200]); items_pro = []
print(len(items_mmlu), 'MMLU questions,', len(items_pro), 'MMLU-Pro questions')

In [ ]:
@torch.no_grad()
def run(items, tag):
    ok = []; t0 = time.time()
    for j, (ids, n_opt, gold) in enumerate(items):
        ids = ids.to('cuda:0')
        with torch.autocast('cuda', dtype=DT):
            try: lg = model(input_ids=ids, logits_to_keep=1).logits[0, -1]
            except TypeError: lg = model(input_ids=ids).logits[0, -1]
        lg = lg.float(); pred = int(torch.argmax(torch.stack([torch.logsumexp(lg[torch.tensor(v, device=lg.device)], 0) for v in let_var[:n_opt]])).item()); ok.append(int(pred == gold))
        if (j + 1) % 50 == 0: print(f'  {tag}: {j+1}/{len(items)}  acc so far {sum(ok)/len(ok):.3f}  ({(time.time()-t0)/60:.1f} min)')
    return ok

def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'),) * 2
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

def sign_test(b, c):          # exact two-sided sign test on the questions where the two models differ
    n = b + c
    if n == 0: return 1.0
    k = min(b, c); p = sum(math.comb(n, i) for i in range(0, k + 1)) / 2**n
    return min(1.0, 2 * p)

results = {}
for name, items in (('MMLU', items_mmlu), ('MMLU-Pro', items_pro)):
    if not items: continue
    with model.disable_adapter(): base = run(items, name + ' base')
    tuned = run(items, name + ' tuned')
    n = len(items); kb, kt = sum(base), sum(tuned)
    only_t = sum(1 for x, y in zip(base, tuned) if y and not x); only_b = sum(1 for x, y in zip(base, tuned) if x and not y)
    lb, ub = wilson(kb, n); lt, ut = wilson(kt, n)
    results[name] = dict(n=n, base=kb/n, base_ci=[lb, ub], tuned=kt/n, tuned_ci=[lt, ut], only_tuned_right=only_t, only_base_right=only_b, sign_test_p=sign_test(only_t, only_b))
    print(f"\n{name} (n={n}): base {kb/n:.3f} [{lb:.3f}, {ub:.3f}] | tuned {kt/n:.3f} [{lt:.3f}, {ut:.3f}] | only tuned right {only_t}, only base right {only_b}, sign-test p={results[name]['sign_test_p']:.3f}")
os.makedirs(f'{GDIR}/results', exist_ok=True)
json.dump(dict(model=MODEL_ID, adapter=ADAPTER, time=time.strftime('%Y-%m-%d %H:%M'), results=results), open(f'{GDIR}/results/eval_{time.strftime("%Y%m%d_%H%M")}.json', 'w'), indent=1)
print('\nsaved to', f'{GDIR}/results/')

## How to read the result
- If the two intervals overlap heavily and the sign-test p-value is above 0.05, the adapter did **not** measurably change MMLU on this sample. That is an honest result, not a failure of the notebook.
- A gain on MMLU-Pro from MMLU-style training would be a bonus; no change is the expected outcome.
- For a tighter estimate raise `N_MMLU` / `N_PRO` (each model needs roughly 1 second per question on two T4s - an estimate, check the timing lines).